In [1]:
from pyspark.sql import SparkSession

import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


spark = SparkSession.builder\
    .appName("Spark Aggregation Functions")\
    .getOrCreate()

In [2]:
listings = spark.read.csv(
    "../data/listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)
listings.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [3]:
reviews = spark.read.csv(
    "../data/reviews.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)
reviews.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)



In [4]:
# 1. For each listing compute string category depending on its price, and add it as a new column.
# A category is defined in the following way:
#
# * price < 50 -> "Budget"
# * 50 <= price < 150 -> "Mid-range"
# * price >= 150 -> "Luxury"
#
# Only include listings where the price is not null.
# Count the number of listings in each category

from pyspark.sql.functions import regexp_replace, when, count


listings = listings.withColumn('price_numeric', regexp_replace('price', '[\$,]', '').cast('float'))

# TODO: Implement a UDF
# TODO: Apply the UDF to create a new DataFrame

listings.printSchema()

listing_by_category = listings.filter(listings.price_numeric.isNotNull())\
    .withColumn(
    "category",
    when(listings.price_numeric < 50, "Budget")\
    .when(listings.price_numeric < 150, "Mid-range")\
    .otherwise("Luxury")
)

listing_by_category.select('id', 'name', 'price_numeric', 'category')\
    .show(truncate=False)

listing_by_category.groupBy('category')\
    .agg(count(listing_by_category.category).alias('count_category'))\
    .orderBy('count_category', ascending=False)\
    .show()


root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [5]:
from pyspark.sql.functions import regexp_replace, udf, count
from pyspark.sql.types import StringType

listings = listings.withColumn('price_numeric', regexp_replace('price', '[\$,]', '').cast('float'))

# TODO: Implement a UDF
# TODO: Apply the UDF to create a new DataFrame

def get_category(price):
    if price is None: return None
    if price < 50: return "Budget"
    if price < 150: return "Mid-range"
    return "Luxury"

categorize_udf = udf(get_category, StringType())

listings_with_category = listings\
    .withColumn("category", categorize_udf("price_numeric"))

listings_with_category.groupBy('category')\
    .agg(count(listings_with_category.category).alias('count_category'))\
    .orderBy('count_category', ascending=False)\
    .show()

+---------+--------------+
| category|count_category|
+---------+--------------+
|Mid-range|         28333|
|   Luxury|         27516|
|   Budget|          6114|
|     NULL|             0|
+---------+--------------+



In [1]:
# 2. In this task you will need to compute a sentiment score per review, and then an average sentiment score per listing.
# A sentiment score indicates how "positive" or "negative" a review is. The higher the score the more positive it is, and vice-versa.
#
# To compute a sentiment score per review compute the number of positive words in a review and subtract the number of negative
# words in the same review (the list of words is already provided)
#
# To complete this task, compute a DataFrame that contains the following fields:
# * name - the name of a Listing
# * average_sentiment - average sentiment of reviews computed using the algorithm described above
from pyspark.sql.types import FloatType
from pyspark.sql.functions import regexp_replace, udf, count

# Lists of positive and negative words
positive_words = {'good', 'great', 'excellent', 'amazing', 'fantastic',
                  'wonderful', 'pleasant', 'lovely', 'nice', 'enjoyed'}
negative_words = {'bad', 'terrible', 'awful', 'horrible', 'disappointing',
                  'poor', 'hate', 'unpleasant', 'dirty', 'noisy'}

# TODO: Implement the UDF
def sentiment_score(comment):
    if comment is None:
            return 0.0
    p_w = 0
    n_w = 0
    words = list(comment.lower().split())
    for word in words:
        if word in positive_words:
            p_w += 1
        elif word in negative_words:
            n_w += 1
    return p_w - n_w


sentiment_score_udf = udf(sentiment_score, FloatType())

reviews_with_sentiment = reviews \
    .withColumn(
        'sentiment_score',
        sentiment_score_udf(reviews.comments)
    )

reviews_with_sentiment.select('id', 'sentiment_score')\
    .orderBy('sentiment_score', ascending=False)\
    .show(50)

# TODO: Create a final DataFrame


NameError: name 'reviews' is not defined